# Trial Demo
Run a trial end-to-end: create a trial, create a fixed route, record frames + risk, optionally append model risk, and generate a video with a risk curve tile.

In [8]:
# Imports
import os
import time
from pathlib import Path

from MIREIA.config import Config
from MIREIA.simulation.world_manager import WorldManager
from MIREIA.simulation.trials import TrialDefinition, EgoTrialConfig, TrialRunner

from MIREIA.analysis.visualizer import compose_trial_comparison_video
from MIREIA.data_collection.dataset_utils import load_jsonl_records
from MIREIA.perception import TemporalInferenceConfig, create_streaming_predictor

In [2]:
# Launch CARLA (skip if already running)
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

In [3]:
trial = TrialDefinition(
    name="demo_trial_runner",
    route_start=(7.409, -64.400, 0.0),
    route_end=(44.003, 50.549, 0.0),
    description="Two ego subtrials on the same world setup.",
    map_name="Town03",
    weather="ClearNoon",
    n_vehicles=30,
    n_pedestrians=20,
    seed=42,
    sync_mode=True,
    fixed_delta=0.05,
)

ego_configs = [
    EgoTrialConfig(
        name="base_speed",
        ego_blueprint="vehicle.lincoln.mkz_2020",
        target_speed_kmh=20.0,
        speed_multiplier=1.0,
        # If ego_camera_position is None and this is True, use vehicle defaults.
        use_vehicle_camera_defaults=True,
        controller_mode="behavior_agent",
        controller_behavior="normal",
    ),
    EgoTrialConfig(
        name="fast_speed",
        ego_blueprint="vehicle.lincoln.mkz_2020",
        target_speed_kmh=20.0,
        speed_multiplier=1.5,
        use_vehicle_camera_defaults=True,
        controller_mode="behavior_agent",
        controller_behavior="normal",
    ),
]

runner = TrialRunner(verbose=True)

In [ ]:
summaries = runner.run_trial(
    trial=trial,
    ego_configs=ego_configs,
    max_steps=6000,
    image_stride=5,                             # save an RGB frame every 5 ticks
    store_topdown_images=True,                  # save top-down images
    topdown_resolution=(1024, 1024),              # top-down camera output size
    topdown_fov=95.0,                           # top-down camera field of view
    topdown_align_risk_rotation=True,           # True => rotated/aligned with risk map
    store_risk_frame_images=True,               # save dynamic risk map image every tick
    store_static_risk_map=False,                # disable single static map
    draw_debug_every_tick=True,                 # keep debug visuals enabled
    draw_debug_skip_after_capture_ticks=1,      # skip capture tick and next tick => draw 3/5
    draw_debug_skip_before_capture_ticks=0,
 )

print("\n=== Trial Summary ===")
for s in summaries:
    print(
        f"subtrial={s.subtrial_name} "
        f"frames={s.num_frames} "
        f"duration_s={s.duration_s:.2f} "
        f"distance_m={s.traveled_m:.2f} "
        f"risk_auc={s.risk_auc:.4f} "
        f"risk_per_meter={s.risk_per_meter:.6f} "
        f"finished={s.finished}"
    )
    print(f"summary_path={s.run_path}/summary.json")

Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town10HD_Opt'.
Loading scenario 'trial_demo_trial_runner__base_speed' (map=Town03, seed=42)...
Switching map from 'Carla/Maps/Town10HD_Opt' to 'Town03'...
Weather set to 'ClearNoon'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Spawned 30 / 30 requested vehicles.
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of coll

: 

: 

In [ ]:
# Real-time streaming inference (FIFO 16 features, burn-in 12, evaluate last 4)

checkpoint_path = Path(Config.PATH_TO_MODELS) / "seq2seq_risk_checkpoint.pt"  # adjust to your checkpoint name
if checkpoint_path.exists():
    streaming_predictor = create_streaming_predictor(
        checkpoint_path=str(checkpoint_path),
        temporal_config=TemporalInferenceConfig(sequence_len=16, burn_in_frames=12, eval_frames=4),
        strict=False,
    )

    trial_stream = TrialDefinition(
        name=f"{trial.name}_streaming",
        route_start=trial.route_start,
        route_end=trial.route_end,
        description="Streaming inference demo run (real-time predictor).",
        map_name=trial.map_name,
        weather=trial.weather,
        n_vehicles=trial.n_vehicles,
        n_pedestrians=trial.n_pedestrians,
        seed=trial.seed,
        sync_mode=trial.sync_mode,
        fixed_delta=trial.fixed_delta,
    )

    stream_summaries = runner.run_trial(
        trial=trial_stream,
        ego_configs=[ego_configs[0]],
        max_steps=2000,
        image_stride=Config.RECORD_EVERY_N_TICKS,
        store_topdown_images=False,
        store_risk_frame_images=False,
        streaming_predictor=streaming_predictor,
    )
    print("Streaming run created:", stream_summaries[0].run_path)
else:
    print(f"Skipping streaming demo: checkpoint not found at {checkpoint_path}")



Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town03'.
Loading scenario 'trial_demo_trial_runner_streaming__base_speed' (map=Town03, seed=42)...
Weather set to 'ClearNoon'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Spawned 30 / 30 requested vehicles.
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of c

In [9]:
summary1_path = r"D:\\Projectes\\TFG\\MIREIA\\trials\\demo_trial_runner\\runs\\20260328_235718_base_speed\\"
summary2_path = r"D:\\Projectes\\TFG\\MIREIA\\trials\\demo_trial_runner_streaming\\runs\\20260329_112130_base_speed\\"

# Load summaries into TrialSummary objects (adjust paths if needed)
from MIREIA.simulation.trials import TrialRunSummary
summaries = []
for path in [summary1_path, summary2_path]:
    if Path(path).exists():
        summaries.append(TrialRunSummary.load_from_json(path))
    else:
        print(f"Summary JSON not found at {path}")

# Compare two trials (uses the two runs from the cell above if available).
if len(summaries) >= 2:
    jsonl_a = Path(summaries[0].run_path) / "dataset.jsonl"
    jsonl_b = Path(summaries[1].run_path) / "dataset.jsonl"
    comparison_video = compose_trial_comparison_video(
        jsonl_path_a=str(jsonl_a),
        jsonl_path_b=str(jsonl_b),
        output_name="trial_comparison_video.mp4",
        fps=Config.RECORDING_FPS,
        label_a=summaries[0].subtrial_name,
        label_b=summaries[1].subtrial_name,
        calculated_risk_key="predicted_risk",  # optional; gracefully ignored if absent
    )
    print(f"Comparison video saved to: {comparison_video}")
else:
    print("Need at least 2 trial summaries to run comparison example.")

# Backwards compatibility check against an existing legacy scenario JSONL.
legacy_jsonl = Path(Config.PATH_TO_SCENARIOS) / "01A_ClearNoon_Town01_HighVol" / "dataset.jsonl"
if legacy_jsonl.exists():
    legacy_records = load_jsonl_records(str(legacy_jsonl))
    first = legacy_records[0] if legacy_records else {}
    required_keys = ["frame_id", "rgb_image_path", "ground_truth_risk", "ego", "environment"]
    missing = [k for k in required_keys if k not in first]
    if missing:
        print("Legacy JSONL warning, missing keys:", missing)
    else:
        print("Legacy JSONL compatibility check passed.")
else:
    print(f"Legacy JSONL not found at {legacy_jsonl}")

Comparison video saved to: d:\Projectes\TFG\MIREIA\trials\demo_trial_runner\runs\20260328_235718_base_speed\trial_comparison_video.mp4
Legacy JSONL compatibility check passed.


In [12]:
from pathlib import Path

from MIREIA.analysis.visualizer import compose_trial_dataset_video

run_dir = Path(r"D:\\Projectes\\TFG\\MIREIA\\trials\\demo_trial_runner_streaming\\runs\\20260329_112130_base_speed\\")
jsonl_path = run_dir / "dataset.jsonl"
if not jsonl_path.exists():
    raise FileNotFoundError(f"Dataset JSONL not found: {jsonl_path}")

video_path = compose_trial_dataset_video(
    jsonl_path=str(jsonl_path),
    output_name="trial_dataset_video_with_risk_graph.mp4",
    fps=10,
    calculated_risk_key="predicted_risk",  # ignored if key is missing in records
)
print(f"Saved video to: {video_path}")

Saved video to: D:\Projectes\TFG\MIREIA\trials\demo_trial_runner_streaming\runs\20260329_112130_base_speed\trial_dataset_video_with_risk_graph.mp4


stop here


In [ ]:
stop

In [ ]:
import random
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

# reload packages (for development)
import importlib

# Direct run: spawn ego + attach controller (no Scenario, no WorldManager)
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()

# Keep previous world settings to restore later
prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points to run a route.")

# Randomize ego spawn position each execution
ego_spawn_index = random.randrange(len(spawn_points))

handler = TrafficHandler(client, world, seed=42)
base_speed_kmh = 20.0
controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

# Demo: modify target speed every tick from the base speed.
# Signature: fn(base_speed_kmh, tick_index, controller) -> target_speed_kmh
def demo_speed_modifier(base_speed, tick_idx, _controller):
    phase = (tick_idx // 120) % 3
    if phase == 0:
        return 0.5 * base_speed   # slower block
    if phase == 1:
        return 1.0 * base_speed   # nominal block
    return 2.0 * base_speed       # faster block

controller.set_speed_modifier(demo_speed_modifier)

ego = handler.spawn_ego(
    blueprint_id="vehicle.lincoln.mkz_2020",
    spawn_index=ego_spawn_index,
    autopilot=False,
    controller=controller,
 )

if hasattr(carla, "VehicleLightState"):
    light_state = (
        carla.VehicleLightState.Position
        | carla.VehicleLightState.LowBeam
        | carla.VehicleLightState.Interior
    )
    ego.set_light_state(carla.VehicleLightState(light_state))

# Let CARLA settle one frame so ego transform and waypoint are consistent
world.tick()

start = ego.get_location()

# Build valid destination candidates, then choose one randomly each execution
candidates = []
for i, sp in enumerate(spawn_points):
    if i == ego_spawn_index:
        continue
    d = start.distance(sp.location)
    if d > 35.0:
        candidates.append((d, i))

if not candidates:
    raise RuntimeError("No suitable destination spawn point found.")

_, best_idx = random.choice(candidates)
end = spawn_points[best_idx].location
controller.set_destination(start, end)
print(f"Plan length: {controller.get_plan_length()} waypoints")

print(f"Spawn index: {ego_spawn_index}")
print(f"Start: ({start.x:.1f}, {start.y:.1f})")
print(f"End:   ({end.x:.1f}, {end.y:.1f})  [spawn index {best_idx}]")
print("Speed profile phases: 0.5x -> 1.0x -> 2.0x (changes every 120 ticks)")

# Risk setup with WorldManager: bake static first, then query per-tick risk
bridge = SimulationBridge(world)
wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
wm_risk.world = world
wm_risk.bridge = bridge
wm_risk._WorldManager__bake_static_risk_map()
risk_values = []

def draw_ego_glow(vehicle: carla.Vehicle):
    t = vehicle.get_transform()
    loc = t.location
    extent = vehicle.bounding_box.extent
    world.debug.draw_box(
        box=carla.BoundingBox(loc, extent),
        rotation=t.rotation,
        thickness=0.15,
        color=carla.Color(0, 255, 255),
        life_time=0.08,
    )
    world.debug.draw_point(
        location=carla.Location(x=loc.x, y=loc.y, z=loc.z + 1.0),
        size=0.2,
        color=carla.Color(255, 255, 0),
        life_time=0.08,
    )

def draw_route_endpoints(start_loc: carla.Location, end_loc: carla.Location):
    world.debug.draw_point(
        location=carla.Location(x=start_loc.x, y=start_loc.y, z=start_loc.z + 0.5),
        size=0.25,
        color=carla.Color(255, 0, 0),
        life_time=0.08,
    )
    world.debug.draw_point(
        location=carla.Location(x=end_loc.x, y=end_loc.y, z=end_loc.z + 0.5),
        size=0.25,
        color=carla.Color(255, 0, 0),
        life_time=0.08,
    )

for step in range(10000):
    handler.run_ego_controller_step()
    world.tick()
    bridge.update()
    try:
        risk_now = wm_risk.get_risk()
        risk_values.append(float(risk_now))
    except RuntimeError:
        pass

    draw_ego_glow(ego)
    draw_route_endpoints(start, end)
    controller.draw_plan(world, max_points=1400, life_time=0.08)
    if step % 60 == 0:
        print(f"step={step:04d} target_speed={controller.get_last_applied_target_speed():.2f} km/h")
    if controller.done():
        print(f"Route finished at step {step}")
        for i in range(100):
            draw_ego_glow(ego)
            draw_route_endpoints(start, end)
            world.tick()
        break

dt = settings.fixed_delta_seconds or 0.05
if risk_values:
    total_risk_auc = float(np.trapz(np.array(risk_values, dtype=np.float64), dx=dt))
    print(f"Total route risk AUC: {total_risk_auc:.4f}")
    print(f"Risk samples: {len(risk_values)}  duration: {len(risk_values) * dt:.2f}s")
else:
    print("No risk samples collected.")

handler.destroy_all()
world.apply_settings(prev_settings)
print("Done.")

RuntimeError: time-out of 15000ms while waiting for the simulator, make sure the simulator is ready and connected to 127.0.0.1:2000

In [ ]:
# Fixed-route comparison: same manual route (by carla.Location), different speed multipliers (0.5x vs 2.0x)
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()
spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points for fixed-route comparison.")

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

# Manual route definition by coordinates.
# You can use carla.Location(...) or carla.Transform(carla.Location(...), carla.Rotation()).
START_CARLA = carla.Location(x=7.409, y=-64.400, z=0.000)
END_CARLA = carla.Location(x=44.003, y=50.549, z=0.000)

def as_location(value):
    if isinstance(value, carla.Transform):
        return value.location
    if isinstance(value, carla.Location):
        return value
    raise TypeError("START_CARLA and END_CARLA must be carla.Location or carla.Transform")

raw_start_loc = as_location(START_CARLA)
raw_end_loc = as_location(END_CARLA)

# Project points to driving lane to get route-ready points and safe spawn orientation.
start_wp = world_map.get_waypoint(
    carla.Location(x=raw_start_loc.x, y=raw_start_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
end_wp = world_map.get_waypoint(
    carla.Location(x=raw_end_loc.x, y=raw_end_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
if start_wp is None or end_wp is None:
    raise RuntimeError("Could not project START_CARLA/END_CARLA to a driving lane")

start_tf = start_wp.transform
start_tf.location.z += 0.5
start_loc = start_tf.location
end_loc = end_wp.transform.location

print(f"Manual route input start: ({raw_start_loc.x:.3f}, {raw_start_loc.y:.3f}, {raw_start_loc.z:.3f})")
print(f"Manual route input end:   ({raw_end_loc.x:.3f}, {raw_end_loc.y:.3f}, {raw_end_loc.z:.3f})")
print(f"Projected start: ({start_loc.x:.3f}, {start_loc.y:.3f}, {start_loc.z:.3f})")
print(f"Projected end:   ({end_loc.x:.3f}, {end_loc.y:.3f}, {end_loc.z:.3f})")

def nearest_spawn_indices(target_loc: carla.Location) -> list[int]:
    dists = [(target_loc.distance(sp.location), i) for i, sp in enumerate(spawn_points)]
    dists.sort(key=lambda x: x[0])
    return [i for _, i in dists]

def run_fixed_route(multiplier: float, base_speed_kmh: float = 20.0, max_steps: int = 10000) -> dict:
    handler = TrafficHandler(client, world, seed=42)
    controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

    def speed_modifier(base_speed, tick_idx, _controller):
        return multiplier * base_speed

    controller.set_speed_modifier(speed_modifier)

    ego = None
    try:
        ego = handler.spawn_ego(
            blueprint_id="vehicle.lincoln.mkz_2020",
            spawn_point=start_tf,
            autopilot=False,
            controller=controller,
        )
    except RuntimeError as exc:
        print(f"Exact transform spawn failed: {exc}")

    if ego is None:
        for fallback_idx in nearest_spawn_indices(start_loc)[:20]:
            try:
                ego = handler.spawn_ego(
                    blueprint_id="vehicle.lincoln.mkz_2020",
                    spawn_index=fallback_idx,
                    autopilot=False,
                    controller=controller,
                )
                print(f"Spawn fallback succeeded at spawn_index={fallback_idx}")
                break
            except RuntimeError:
                continue

    if ego is None:
        raise RuntimeError("Failed to spawn ego vehicle: start position is blocked. Try another START_CARLA.")

    if hasattr(carla, "VehicleLightState"):
        light_state = (
            carla.VehicleLightState.Position
            | carla.VehicleLightState.LowBeam
            | carla.VehicleLightState.Interior
        )
        ego.set_light_state(carla.VehicleLightState(light_state))

    world.tick()
    start = ego.get_location()
    controller.set_destination(start, end_loc)

    bridge = SimulationBridge(world)
    wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
    wm_risk.world = world
    wm_risk.bridge = bridge
    wm_risk._WorldManager__bake_static_risk_map()

    risks = []
    positions = []
    finished_step = None

    for step in range(max_steps):
        handler.run_ego_controller_step()
        world.tick()
        bridge.update()

        ego_kin = bridge.get_ego_kinematics()
        if ego_kin is not None:
            positions.append((float(ego_kin.x), float(ego_kin.y)))

        try:
            risks.append(float(wm_risk.get_risk()))
        except RuntimeError:
            pass

        # Visualize current planned path (same style as Cell 4).
        controller.draw_plan(world, max_points=1400, life_time=0.08)

        if controller.done():
            finished_step = step
            break

    # Traveled distance (meters) from sampled ego trajectory.
    if len(positions) >= 2:
        pos_arr = np.array(positions, dtype=np.float64)
        seg = np.linalg.norm(np.diff(pos_arr, axis=0), axis=1)
        traveled_m = float(seg.sum())
    else:
        seg = np.array([], dtype=np.float64)
        traveled_m = 0.0

    # Distance-integrated risk: sum(average segment risk * segment length).
    risk_arr = np.array(risks, dtype=np.float64) if risks else np.array([], dtype=np.float64)
    if seg.size > 0 and risk_arr.size >= 2:
        n = min(seg.size, risk_arr.size - 1)
        risk_integral_distance = float(np.sum(0.5 * (risk_arr[:n] + risk_arr[1:n + 1]) * seg[:n]))
    else:
        risk_integral_distance = float("nan")

    risk_per_meter = (risk_integral_distance / traveled_m) if traveled_m > 0 and np.isfinite(risk_integral_distance) else float("nan")
    dt = settings.fixed_delta_seconds or 0.05

    handler.destroy_all()

    return {
        "multiplier": multiplier,
        "target_speed_kmh": multiplier * base_speed_kmh,
        "steps": finished_step if finished_step is not None else max_steps,
        "num_samples": len(risks),
        "duration_s": len(risks) * dt,
        "traveled_m": traveled_m,
        "risk_integral_distance": risk_integral_distance,
        "risk_per_meter": risk_per_meter,
    }

results = []
for m in (0.5, 2.0):
    res = run_fixed_route(m, base_speed_kmh=20.0)
    results.append(res)
    print(
        f"mult={res['multiplier']:.1f}x  target={res['target_speed_kmh']:.1f} km/h  "
        f"risk/m={res['risk_per_meter']:.6f}  distance={res['traveled_m']:.2f}m  "
        f"samples={res['num_samples']}  duration={res['duration_s']:.2f}s"
    )

if len(results) == 2:
    a = results[0]["risk_per_meter"]
    b = results[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

world.apply_settings(prev_settings)
print("Fixed-route comparison done.")

Manual route input start: (7.409, -64.400, 0.000)
Manual route input end:   (44.003, 50.549, 0.000)
Projected start: (7.409, -64.400, 0.500)
Projected end:   (44.003, 50.549, 0.000)
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Destroyed ego vehicle.
mult=0.5x  target=10.0 km/h  risk/m=1.572004  distance=364.67m  samples=3495  duration=174.75s
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Destroyed ego vehicle.
mult=2.0x  target=40.0 km/h  risk/m=2.122821  distance=362.43m  samples=1056  duration=52.80s
risk/m ratio (2.0x / 0.5x): 1.3504
Fixed-route comparison done.


In [ ]:
# Same fixed-route comparison, but with extra traffic before route execution
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

if "START_CARLA" not in globals() or "END_CARLA" not in globals():
    raise RuntimeError("Run Cell 5 first so START_CARLA and END_CARLA are defined.")

def as_location(value):
    if isinstance(value, carla.Transform):
        return value.location
    if isinstance(value, carla.Location):
        return value
    raise TypeError("START_CARLA and END_CARLA must be carla.Location or carla.Transform")

client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()
spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points for fixed-route comparison.")

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

raw_start_loc = as_location(START_CARLA)
raw_end_loc = as_location(END_CARLA)

start_wp = world_map.get_waypoint(
    carla.Location(x=raw_start_loc.x, y=raw_start_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
end_wp = world_map.get_waypoint(
    carla.Location(x=raw_end_loc.x, y=raw_end_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
if start_wp is None or end_wp is None:
    raise RuntimeError("Could not project START_CARLA/END_CARLA to a driving lane")

start_tf = start_wp.transform
start_tf.location.z += 0.5
start_loc = start_tf.location
end_loc = end_wp.transform.location

print(f"Using START_CARLA: ({start_loc.x:.3f}, {start_loc.y:.3f}, {start_loc.z:.3f})")
print(f"Using END_CARLA:   ({end_loc.x:.3f}, {end_loc.y:.3f}, {end_loc.z:.3f})")

N_TRAFFIC_VEHICLES = 45
BASE_SPEED_KMH = 20.0

def nearest_spawn_indices(target_loc: carla.Location) -> list[int]:
    dists = [(target_loc.distance(sp.location), i) for i, sp in enumerate(spawn_points)]
    dists.sort(key=lambda x: x[0])
    return [i for _, i in dists]

def run_fixed_route_with_traffic(multiplier: float, max_steps: int = 10000) -> dict:
    handler = TrafficHandler(client, world, seed=42)
    controller = SimpleRouteController(target_speed=BASE_SPEED_KMH, sampling_resolution=2.0)

    def speed_modifier(base_speed, tick_idx, _controller):
        return multiplier * base_speed

    controller.set_speed_modifier(speed_modifier)

    ego = None
    try:
        ego = handler.spawn_ego(
            blueprint_id="vehicle.lincoln.mkz_2020",
            spawn_point=start_tf,
            autopilot=False,
            controller=controller,
        )
    except RuntimeError as exc:
        print(f"Exact transform spawn failed: {exc}")

    if ego is None:
        for fallback_idx in nearest_spawn_indices(start_loc)[:20]:
            try:
                ego = handler.spawn_ego(
                    blueprint_id="vehicle.lincoln.mkz_2020",
                    spawn_index=fallback_idx,
                    autopilot=False,
                    controller=controller,
                )
                print(f"Spawn fallback succeeded at spawn_index={fallback_idx}")
                break
            except RuntimeError:
                continue

    if ego is None:
        raise RuntimeError("Failed to spawn ego vehicle: start position is blocked. Try another START_CARLA.")

    # Add extra background traffic before route execution.
    spawned_traffic = handler.spawn_vehicles(n=N_TRAFFIC_VEHICLES, safe=True, car_lights_on=False)
    print(f"Spawned traffic vehicles: {len(spawned_traffic)}")

    if hasattr(carla, "VehicleLightState"):
        light_state = (
            carla.VehicleLightState.Position
            | carla.VehicleLightState.LowBeam
            | carla.VehicleLightState.Interior
        )
        ego.set_light_state(carla.VehicleLightState(light_state))

    world.tick()
    start = ego.get_location()
    controller.set_destination(start, end_loc)

    bridge = SimulationBridge(world)
    wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
    wm_risk.world = world
    wm_risk.bridge = bridge
    wm_risk._WorldManager__bake_static_risk_map()

    risks = []
    positions = []
    finished_step = None

    for step in range(max_steps):
        handler.run_ego_controller_step()
        world.tick()
        bridge.update()

        ego_kin = bridge.get_ego_kinematics()
        if ego_kin is not None:
            positions.append((float(ego_kin.x), float(ego_kin.y)))

        try:
            risks.append(float(wm_risk.get_risk()))
        except RuntimeError:
            pass

        # Visualize current planned path (same style as Cell 4).
        controller.draw_plan(world, max_points=1400, life_time=0.08)

        if controller.done():
            finished_step = step
            break

    if len(positions) >= 2:
        pos_arr = np.array(positions, dtype=np.float64)
        seg = np.linalg.norm(np.diff(pos_arr, axis=0), axis=1)
        traveled_m = float(seg.sum())
    else:
        seg = np.array([], dtype=np.float64)
        traveled_m = 0.0

    risk_arr = np.array(risks, dtype=np.float64) if risks else np.array([], dtype=np.float64)
    if seg.size > 0 and risk_arr.size >= 2:
        n = min(seg.size, risk_arr.size - 1)
        risk_integral_distance = float(np.sum(0.5 * (risk_arr[:n] + risk_arr[1:n + 1]) * seg[:n]))
    else:
        risk_integral_distance = float("nan")

    risk_per_meter = (risk_integral_distance / traveled_m) if traveled_m > 0 and np.isfinite(risk_integral_distance) else float("nan")
    dt = settings.fixed_delta_seconds or 0.05

    handler.destroy_all()

    return {
        "multiplier": multiplier,
        "target_speed_kmh": multiplier * BASE_SPEED_KMH,
        "steps": finished_step if finished_step is not None else max_steps,
        "num_samples": len(risks),
        "duration_s": len(risks) * dt,
        "traveled_m": traveled_m,
        "risk_integral_distance": risk_integral_distance,
        "risk_per_meter": risk_per_meter,
    }

results_traffic = []
for m in (0.5, 2.0):
    res = run_fixed_route_with_traffic(m)
    results_traffic.append(res)
    print(
        f"[traffic] mult={res['multiplier']:.1f}x  target={res['target_speed_kmh']:.1f} km/h  "
        f"risk/m={res['risk_per_meter']:.6f}  distance={res['traveled_m']:.2f}m  "
        f"samples={res['num_samples']}  duration={res['duration_s']:.2f}s"
    )

if len(results_traffic) == 2:
    a = results_traffic[0]["risk_per_meter"]
    b = results_traffic[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"[traffic] risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

world.apply_settings(prev_settings)
print("Fixed-route comparison with extra traffic done.")

Using START_CARLA: (7.409, -64.400, 0.500)
Using END_CARLA:   (44.003, 50.549, 0.000)
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
ERROR: Spawn failed because of collision at spawn position
Spawned 44 / 45 requested vehicles.
Spawned traffic vehicles: 44
Tailgating, moving to the right!
Destroyed 44 vehicles.
Destroyed ego vehicle.
[traffic] mult=0.5x  target=10.0 km/h  risk/m=3.595165  distance=86.89m  samples=1323  duration=66.15s
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
ERROR: Spawn failed because of collision at spawn position
Spawned 44 / 45 requested vehicles.
Spawned traffic vehicles: 44
Destroyed 44 vehicles.
Destroyed ego vehicle.
[traffic] mult=2.0x  target=40.0 km/h  risk/m=3.246295  distance=362.31m  samples=1154  duration=57.70s
[traffic] risk/m ratio (2.0x / 0.5x): 0.9030
Fixed-route comparison with extra traffic done.


In [ ]:
stop

NameError: name 'stop' is not defined

## 1) Create or load a Trial
A Trial owns its folder under `PATH_TO_TRIALS` and references a `route.json` that contains waypoint IDs.

In [ ]:
trial_name = "demo_trial_001"
trial = Trial(
    name=trial_name,
    map_name="Town03",
    weather="ClearNoon",
    description="Demo trial with predefined route.",
    n_vehicles=0,
    n_pedestrians=0,
 )
trial.save()
print(f"Trial saved at: {trial.folder_path}")
print(f"Route JSON: {trial.route_path}")

Trial saved at: d:\Projectes\TFG\MIREIA\trials\demo_trial_001
Route JSON: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\route.json


## 2) Create a route (run once)
Use the interactive picker to select waypoints and save `route.json`. Skip this if the route already exists.

wm = WorldManager(verbose=True)
wm.load_scenario(trial)

route = create_route_from_waypoints(wm.bridge.get_waypoints())
save_route(route, trial.route_path)
plot_route(wm.bridge.get_waypoints(), [wp.id for wp in route.waypoints])

wm.destroy()
print(f"Route saved to: {trial.route_path}")

In [ ]:
run_id, run_path = trial.create_run_folder()
run_path = Path(run_path)
dataset_jsonl = run_path / "dataset.jsonl"
images_dir = run_path / "images"
images_dir.mkdir(parents=True, exist_ok=True)

static_meta = {
    "trial_name": trial.name,
    "run_id": run_id,
}
print(f"Run folder: {run_path}")
print(f"JSONL path: {dataset_jsonl}")

Run folder: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836
JSONL path: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


## 4) Run a trial
The current flow uses autopilot only. A route-following controller can be added later.

In [ ]:
use_controller = False  # False = autopilot
compute_model_risk = False
num_frames = 200
frame_stride = 5

wm = WorldManager(verbose=True)
wm.load_scenario(trial)
wm.enable_recording(
    append=False,
    include_topdown=False,
    include_static_risk_image=False,
    jsonl_path=str(dataset_jsonl),
    static_meta=static_meta,
 )
wm.setup_sensors(save_dir=str(images_dir), enable_map_camera=False)
delta_seconds = wm.world.get_settings().fixed_delta_seconds or 0.05

start_time = time.time()
for frame_id in range(num_frames):
    if frame_id % frame_stride != 0:
        wm.tick()
        continue

    rgb_path = images_dir / f"rgb_{frame_id:06d}.png"
    rel_rgb_path = str(rgb_path.relative_to(run_path))

    def tick_and_log():
        ctrl = wm.ego_vehicle.get_control() if wm.ego_vehicle is not None else None
        extra_fields = {
            "control": {
                "throttle": ctrl.throttle if ctrl else 0.0,
                "steer": ctrl.steer if ctrl else 0.0,
                "brake": ctrl.brake if ctrl else 0.0,
            },
            "mode": "controller" if use_controller else "autopilot",
        }
        wm.tick(
            ground_truth_risk=None,
            rgb_image_path=rel_rgb_path,
            extra_fields=extra_fields,
        )

    wm.sensor_manager.save_ego_frame(save_path=str(rgb_path), tick_fn=tick_and_log)

end_time = time.time()
elapsed = end_time - start_time
trial.add_run_summary(run_id, {"elapsed_seconds": elapsed, "frames": num_frames, "delta_seconds": delta_seconds})
print(f"Trial run completed in {elapsed:.1f}s")

wm.destroy()

Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town10HD_Opt'.
Loading scenario 'demo_trial_001' (map=Town03, seed=42)...
Switching map from 'Carla/Maps/Town10HD_Opt' to 'Town03'...
Weather set to 'ClearNoon'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=True)
SimulationBridge initialized: SimulationBridge(Ego: None, Dynamic Obstacles: 0, Static Obstacles: 7092, EnvState: (Visibility: 282.0m, Friction: 0.80))
Computing static risk map with bounds: (49.96485900878906, 0.818756103515625, 456.9049987792969) and resolution: 2.0m...
Scenario 'demo_trial_001' is ready.
Recording enabled → d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


AttributeError: 'NoneType' object has no attribute 'x'

## 5) Append model risk (optional)
Run post-inference to add `model_risk` per frame. Skip for autopilot-only runs.

## 8) TrialRunner Demo (new trial API)
Run two subtrials with fixed world settings and different ego configs, then print summary metrics and saved paths.